# Partitioned Iceberg Tables

This notebook demonstrates partitioning strategies for Iceberg tables including:
- Identity partitioning
- Time-based partitioning (year, month, day, hour)
- Bucket partitioning
- Truncate partitioning

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_TESTING;
USE SCHEMA PUBLIC;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
-- Identity partition by event_type
CREATE OR REPLACE ICEBERG TABLE EVENTS_BY_TYPE (
    event_id INT,
    event_type STRING,
    event_timestamp TIMESTAMP_NTZ(9),
    event_data VARIANT
)
PARTITION BY (event_type)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

In [ ]:
-- Time-based partition by month
CREATE OR REPLACE ICEBERG TABLE EVENTS_BY_MONTH (
    event_id INT,
    event_type STRING,
    event_timestamp TIMESTAMP_NTZ(9),
    event_data VARIANT
)
PARTITION BY (MONTH(event_timestamp))
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

In [ ]:
-- Bucket partition for high-cardinality columns
CREATE OR REPLACE ICEBERG TABLE EVENTS_BUCKETED (
    event_id INT,
    customer_id INT,
    event_type STRING,
    event_timestamp TIMESTAMP_NTZ(9),
    event_data VARIANT
)
PARTITION BY (BUCKET(16, customer_id))
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

In [ ]:
-- Insert test data into partitioned tables
INSERT INTO EVENTS_BY_TYPE VALUES
    (1, 'login', '2026-03-11 10:00:00.123456789', PARSE_JSON('{"device": "mobile"}')),
    (2, 'purchase', '2026-03-11 10:15:00.987654321', PARSE_JSON('{"amount": 99.99}')),
    (3, 'login', '2026-03-11 10:30:00.111222333', PARSE_JSON('{"device": "desktop"}'));

In [ ]:
-- Show partition specs for all Iceberg tables
SHOW ICEBERG TABLES IN SCHEMA ICEBERG_TESTING.PUBLIC;

## Partition Transform Reference

| Transform | Description | Example |
|-----------|-------------|---------|
| Identity | Partition by exact value | `PARTITION BY (region)` |
| YEAR | Extract year from timestamp | `PARTITION BY (YEAR(ts))` |
| MONTH | Extract month from timestamp | `PARTITION BY (MONTH(ts))` |
| DAY | Extract day from timestamp | `PARTITION BY (DAY(ts))` |
| HOUR | Extract hour from timestamp | `PARTITION BY (HOUR(ts))` |
| BUCKET | Hash into N buckets | `PARTITION BY (BUCKET(16, id))` |
| TRUNCATE | Truncate to width | `PARTITION BY (TRUNCATE(10, name))` |